# Monotonic Stacks — Advanced
### From templates to hard LeetCode problems

---

## The Two Universal Templates

Memorise these. Nearly every monotonic stack problem is a variation of one of them.

In [ ]:
# Template A — Next Greater Element (Monotonic Decreasing Stack)
def next_greater(nums):
    n = len(nums)
    result = [-1] * n     # default: no NGE
    stack = []            # stores indices

    for i in range(n):
        while stack and nums[stack[-1]] < nums[i]:   # breaks decreasing order
            idx = stack.pop()
            result[idx] = nums[i]                    # record on pop
        stack.append(i)

    return result


# Template B — Next Smaller Element (Monotonic Increasing Stack)
def next_smaller(nums):
    n = len(nums)
    result = [-1] * n
    stack = []            # stores indices

    for i in range(n):
        while stack and nums[stack[-1]] > nums[i]:   # breaks increasing order
            idx = stack.pop()
            result[idx] = nums[i]                    # record on pop
        stack.append(i)

    return result


print(next_greater([2, 1, 5, 3, 7]))    # [5, 5, 7, 7, -1]
print(next_smaller([4, 2, 1, 3, 5]))    # [2, 1, -1, -1, -1]

---

## Part 1 — Previous Greater / Previous Smaller

So far we've found things to the **right**. A small logic shift gives you **previous-direction** answers.

Instead of recording when we pop, we record **before we push** — the stack top at that moment is the answer for `i`.

| Pattern | Direction | Record when |
|---------|-----------|-------------|
| Next Greater Element   | left → right | on pop |
| Next Smaller Element   | left → right | on pop |
| Previous Greater Element | left → right | before push |
| Previous Smaller Element | left → right | before push |

In [ ]:
def previous_greater(nums):
    n = len(nums)
    result = [-1] * n
    stack = []   # monotonic decreasing

    for i in range(n):
        # pop everything smaller-or-equal (they are NOT the answer)
        while stack and nums[stack[-1]] <= nums[i]:
            stack.pop()
        # stack top is now the previous greater element for i
        if stack:
            result[i] = nums[stack[-1]]
        stack.append(i)

    return result


def previous_smaller(nums):
    n = len(nums)
    result = [-1] * n
    stack = []   # monotonic increasing

    for i in range(n):
        while stack and nums[stack[-1]] >= nums[i]:
            stack.pop()
        if stack:
            result[i] = nums[stack[-1]]
        stack.append(i)

    return result


print(previous_greater([2, 1, 5, 3, 7]))   # [-1, 2, -1, 5, -1]
print(previous_smaller([4, 2, 1, 3, 5]))   # [-1, -1, -1, 1, 3]

---

## Part 2 — Circular Arrays (LeetCode 503)

Arrays can be **circular** — the NGE for the last element might be at index 0.  
The trick: iterate `2n` times using modulo. Only push during the first pass.

In [ ]:
# LeetCode 503 — Next Greater Element II
def nextGreaterElements(nums):
    n = len(nums)
    result = [-1] * n
    stack = []

    for i in range(2 * n):       # simulate wrapping around
        real_i = i % n
        while stack and nums[stack[-1]] < nums[real_i]:
            idx = stack.pop()
            result[idx] = nums[real_i]
        if i < n:                # only push during the first pass
            stack.append(real_i)

    return result


# For [1, 2, 1]: 1→2, 2→-1 (no greater in circle), 1→2
print(nextGreaterElements([1, 2, 1]))    # [2, -1, 2]
print(nextGreaterElements([5, 4, 3, 2, 1]))  # [-1, 5, 5, 5, 5]

> **Why 2n?** In a circular array of size n, the farthest an NGE can be is n−1 positions away.  
> By iterating 2n and taking `i % n`, every element gets to "see" every element that comes after it in circular order.

---

## Part 3 — Span Accumulation (LeetCode 901)

**Online Stock Span:** For each day's price, find how many consecutive previous days had price ≤ today.

This is Previous Greater — but instead of just recording the boundary, we **accumulate spans** as we pop.

In [ ]:
# LeetCode 901 — Online Stock Span
class StockSpanner:
    def __init__(self):
        self.stack = []   # stores (price, span) pairs

    def next(self, price: int) -> int:
        span = 1
        # collapse all days with price <= today, absorbing their spans
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]   # absorb span
        self.stack.append((price, span))
        return span


spanner = StockSpanner()
prices = [100, 80, 60, 70, 60, 75, 85]
spans  = [spanner.next(p) for p in prices]
print(list(zip(prices, spans)))
# Expected spans: [1, 1, 1, 2, 1, 4, 6]

> **Span accumulation pattern:** When you pop an element, instead of discarding it, you **add its span** to the current span.  
> This compresses multiple days into one stack entry — powerful for streaming problems.

---

## Part 4 — Largest Rectangle in Histogram (LeetCode 84)

The canonical hard monotonic stack problem. For each bar, find the largest rectangle where that bar is the **minimum height**.

Use a **monotonic increasing stack**. When you pop, you've found the bar that is the minimum for a rectangle bounded by the current index (right) and the new stack top (left).

In [ ]:
# LeetCode 84 — Largest Rectangle in Histogram
def largestRectangleArea(heights):
    stack = []        # monotonic increasing, stores indices
    max_area = 0
    heights = heights + [0]   # sentinel forces all bars to be popped

    for i, h in enumerate(heights):
        while stack and heights[stack[-1]] >= h:
            height = heights[stack.pop()]
            # left boundary = new top of stack (or start of array)
            width = i if not stack else i - stack[-1] - 1
            max_area = max(max_area, height * width)
        stack.append(i)

    return max_area


print(largestRectangleArea([2, 1, 5, 6, 2, 3]))   # 10
print(largestRectangleArea([2, 4]))                # 4

### Width calculation explained

After popping index `p`, the rectangle using `heights[p]` as its height extends:
- **left boundary:** `stack[-1] + 1` (the index just after the new top — which is the first bar taller than `heights[p]` to the left)
- **right boundary:** `i - 1` (just before the current bar that triggered the pop)

So `width = i - stack[-1] - 1`. If the stack is empty, the rectangle spans all the way from index 0, so `width = i`.

---

## Part 5 — Trapping Rain Water (LeetCode 42)

Calculate total water trapped between bars.

The monotonic stack approach tracks **left walls** for each water pocket as it pops.  
Each pop reveals a pocket: the **floor** is the popped element, **left wall** is the new stack top, **right wall** is the current index.

In [ ]:
# LeetCode 42 — Trapping Rain Water
def trap(height):
    stack = []    # monotonic decreasing (indices)
    water = 0

    for i in range(len(height)):
        while stack and height[stack[-1]] < height[i]:
            bottom = stack.pop()           # valley floor
            if not stack:
                break                      # no left wall, no pocket
            left_wall = stack[-1]
            h = min(height[left_wall], height[i]) - height[bottom]
            w = i - left_wall - 1
            water += h * w
        stack.append(i)

    return water


print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))   # 6
print(trap([4, 2, 0, 3, 2, 5]))                       # 9

---

## Part 6 — Sum of Subarray Minimums (LeetCode 907)

Find the sum of `min(subarray)` for every possible subarray. Brute force is O(n²).

**Key insight (contribution technique):** For each element, count how many subarrays have it as the minimum, then multiply.  
Stack stores `(value, count)` — count = how many consecutive subarrays this value is the minimum for.

In [ ]:
# LeetCode 907 — Sum of Subarray Minimums
def sumSubarrayMins(arr):
    MOD = 10**9 + 7
    result = 0
    stack = []   # stores (value, count) pairs

    for num in arr:
        count = 1
        # pop elements >= num (current num is now the minimum)
        while stack and stack[-1][0] >= num:
            val, cnt = stack.pop()
            count += cnt   # absorb their contribution count
        stack.append((num, count))
        # running total: each entry contributes (value × count)
        result += sum(v * c for v, c in stack)

    return result % MOD


print(sumSubarrayMins([3, 1, 2, 4]))   # 17
# Subarrays and mins: [3]→3, [1]→1, [2]→2, [4]→4,
# [3,1]→1, [1,2]→1, [2,4]→2, [3,1,2]→1, [1,2,4]→1, [3,1,2,4]→1 → sum=17

---

## Part 7 — Monotonic Deque: Sliding Window Maximum (LeetCode 239)

Same ordering idea, but use a **deque** (double-ended queue) because we also expire elements from the front.

> Deque vs Stack: A deque is needed when you remove from **both ends** — expired window elements from the front, smaller elements from the back.  
> Whenever you see "sliding window" + "min/max", think monotonic deque.

In [ ]:
from collections import deque

# LeetCode 239 — Sliding Window Maximum
def maxSlidingWindow(nums, k):
    dq = deque()   # stores indices, monotonic decreasing by value
    result = []

    for i in range(len(nums)):
        # remove indices that have fallen outside the window
        while dq and dq[0] < i - k + 1:
            dq.popleft()
        # maintain monotonic decreasing order from the back
        while dq and nums[dq[-1]] < nums[i]:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])   # front is always the window max

    return result


print(maxSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3))   # [3, 3, 5, 5, 6, 7]

---

## Complexity Summary

| Metric | Analysis |
|--------|----------|
| **Time** | O(n) — each element is pushed once and popped at most once. The nested `while` loop is amortised, total pops across the entire run ≤ n. |
| **Space** | O(n) — worst case the stack holds all n indices (e.g. strictly decreasing input `[5, 4, 3, 2, 1]`). |

---

## Decision Tree — Quick Reference

| If the problem asks for... | Reach for... |
|----------------------------|--------------|
| Next greater element (right) | Monotonic **decreasing** stack, left→right, record on pop |
| Next smaller element (right) | Monotonic **increasing** stack, left→right, record on pop |
| Previous greater element (left) | Monotonic **decreasing** stack, left→right, record before push |
| Previous smaller element (left) | Monotonic **increasing** stack, left→right, record before push |
| Circular NGE/NSE | Same as above, iterate `2n` with `i % n` |
| Largest area under histogram | Monotonic **increasing** stack + sentinel `0` at end |
| Trapping rain water | Monotonic **decreasing** stack, 3-pointer pop pattern |
| Sum of subarray min/max | Monotonic **increasing** stack + contribution counting |
| Sliding window min/max | Monotonic **deque** (pop from both ends) |
| Spans / consecutive day count | Monotonic **decreasing** stack, accumulate span on pop |

---

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Storing values instead of indices | Always push `i`, retrieve value as `nums[i]` |
| Wrong comparison (`<` vs `<=`) | Trace `[3, 3, 3]` manually — duplicates expose this fast |
| Forgetting leftover stack elements | Their result stays at default (`-1`, `0`, `n`) — verify it matches the problem |
| Stack when you need a deque | Sliding window needs front removal — use `deque` |

---

## LeetCode Problem Map

| Problem | # | Type | Key Variation |
|---------|---|------|---------------|
| Daily Temperatures | 739 | NGE distance | `result = i - idx` |
| Next Greater Element I | 496 | NGE two arrays | Hashmap after stack pass |
| Next Greater Element II | 503 | NGE circular | Iterate 2n with modulo |
| Online Stock Span | 901 | PGE + span | Accumulate spans on pop |
| Largest Rectangle | 84 | Area / histogram | Width from boundary indices |
| Maximal Rectangle | 85 | 2D histogram | Apply LC84 row by row |
| Trapping Rain Water | 42 | Volume / pocket | 3-pointer pop (floor + walls) |
| Sum of Subarray Mins | 907 | Subarray contribution | Contribution × count |
| Sliding Window Max | 239 | Window max | Monotonic deque |

---

## Practice Problems

Try implementing these without looking at the templates above:

In [ ]:
# 1. Maximal Rectangle (LeetCode 85)
# Given a 2D binary matrix, find the largest rectangle containing only 1s.
# Hint: convert each row into a histogram and apply largestRectangleArea.

def maximalRectangle(matrix):
    if not matrix:
        return 0
    # your code here
    pass


matrix = [
    ["1","0","1","0","0"],
    ["1","0","1","1","1"],
    ["1","1","1","1","1"],
    ["1","0","0","1","0"]
]
print(maximalRectangle(matrix))   # 6

In [ ]:
# 2. Next Greater Element I (LeetCode 496)
# nums1 is a subset of nums2. For each element in nums1,
# find its next greater element in nums2.

def nextGreaterElement(nums1, nums2):
    # your code here
    pass


print(nextGreaterElement([4, 1, 2], [1, 3, 4, 2]))   # [-1, 3, -1]